# 02. Coqui-TTS 本地 XTTS-v2 中文声音克隆

上一章我们用预置音色做了文本转语音，本章更进一步：**用一段真实人声克隆出相同音色**。

**XTTS-v2 工作原理**（三段式流水线）：

1. **Speaker Encoder** —— 从参考音频中提取 512 维说话人向量（音色指纹），参考音频只需 6~30 秒干净的单人说话声
2. **GPT 骨干网络** —— 以「文本 token + 说话人向量」为条件，自回归地生成语音离散特征（latent）
3. **Vocoder 声码器** —— 把 latent 还原为 24kHz 波形

**为什么选择本地加载**：官方 `from_pretrained` 路径会联网下载模型并弹出 CPML 许可确认框，本章直接从本地 `model\XTTS-v2` 目录加载全部权重，完全离线可复现。

**本章任务**：已从 B 站下载了一段吴彦祖的普通话访谈（151 秒），截取其中 18 秒作为参考音频，克隆音色合成任意中文文本。

In [1]:
# Cell 1: 环境准备与兼容性补丁
# 本环境为共享课程环境，transformers 已是 5.x；coqui-tts 0.27 仍引用
# transformers.pytorch_utils.isin_mps_friendly（该函数只服务苹果 MPS 设备，
# 在 CUDA/CPU 上与 torch.isin 语义完全等价），transformers 5.x 已将其移除。
# 在导入 TTS 之前手动补齐即可，无需修改任何库文件。

import transformers.pytorch_utils as _tpu
import torch

if not hasattr(_tpu, "isin_mps_friendly"):
    _tpu.isin_mps_friendly = torch.isin

from TTS.api import TTS  # pip 包名是 coqui-tts，导入名仍是 TTS
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts, XttsAudioConfig

print("torch:", torch.__version__, "| CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 | CUDA 可用: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 1. 从本地目录加载 XTTS-v2

`model\XTTS-v2` 目录下的文件分工：

| 文件 | 作用 | 大小 |
|---|---|---|
| `config.json` | 模型结构 / 音频参数配置 | 4.5 KB |
| `model.pth` | 主模型权重（GPT + Speaker Encoder + Vocoder） | 1.87 GB |
| `vocab.json` | 文本分词表（支持 17 种语言，含 zh-cn） | 377 KB |
| `dvae.pth` | 语音离散自编码器（latent 提取） | 210 MB |
| `speakers_xtts.pth` | 官方预置说话人向量库 | 7.8 MB |
| `mel_stats.pth` | 梅尔频谱归一化统计量 | 1 KB |

加载三步走：读配置 → 初始化结构 → 载入权重（`eval=True` 自动切推理模式并放到可用设备）。

In [2]:
# Cell 2: 本地加载 XTTS-v2（首次运行需读入 1.87GB 权重，约 1~2 分钟）
import os

MODEL_DIR = r"S:\PythonAILearningCourse\08-tts-voice-robot\model\XTTS-v2"

# 1) 从本地 json 读配置（新版 coqpit 的 load_json 是实例方法）
config = XttsConfig()
config.load_json(os.path.join(MODEL_DIR, "config.json"))

# 2) 按配置构建模型结构
model = Xtts.init_from_config(config)

# 3) 加载本地权重，eval=True 切换到推理模式
model.load_checkpoint(config, checkpoint_dir=MODEL_DIR, eval=True)
model.cuda()

n_params = sum(p.numel() for p in model.parameters())
# 模型实际输出的波形采样率在 model_args.output_sample_rate=24000；
# config.audio.sample_rate(22050) 只是历史遗留字段，直接拿去保存会变调
print("模型参数量: %.1fM | 输出采样率: %d Hz" % (n_params / 1e6, config.model_args["output_sample_rate"]))

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 255), got 50256. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 607), got 50256. This may result in unexpected behavior.


模型参数量: 466.9M | 输出采样率: 24000 Hz


## 2. 准备参考音频：声音的"指纹"

声音克隆的质量 80% 取决于参考音频。合格标准：

- **6~30 秒**，单人说话，无背景音乐 / 音效 / 第二人插话
- 干净人声，语速自然（访谈、朗读、独白皆可）
- 格式不限（wav / mp3 / m4a），模型内部会重采样到 22050Hz

本章素材：用 `yt-dlp` 从 B 站搜索下载的吴彦祖访谈（`吴彦祖_采访.wav`，151 秒），
脚本自动选取了 8.5 秒起语音最密集的 **18 秒**片段保存为 `吴彦祖_参考.wav`。

In [3]:
# Cell 3: 可视化参考音频，确认是干净人声
import librosa
import numpy as np
import matplotlib.pyplot as plt

REF_WAV = os.path.join(r"S:\PythonAILearningCourse\08-tts-voice-robot\02.coqui-tts", "wyz1.WAV")

ref_y, ref_sr = librosa.load(REF_WAV, sr=None)  # sr=None 保持原始采样率

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei"]  # 图表中文显示
plt.rcParams["axes.unicode_minus"] = False

plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(ref_y)) / ref_sr, ref_y, linewidth=0.3)
plt.title("吴彦祖参考音频波形（%.1f 秒 @ %d Hz）" % (len(ref_y) / ref_sr, ref_sr))
plt.xlabel("时间（秒）")
plt.ylabel("振幅")
plt.tight_layout()
plt.show()

print("时长: %.1f 秒 | 峰值振幅: %.3f（语音正常范围 0.3~1.0）" % (len(ref_y) / ref_sr, np.abs(ref_y).max()))

时长: 4.3 秒 | 峰值振幅: 0.581（语音正常范围 0.3~1.0）


C:\Users\rehuanoo\AppData\Local\Temp\ipykernel_37536\912025799.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 声音克隆合成

`model.inference()` 的关键参数：

| 参数 | 建议值 | 说明 |
|---|---|---|
| `language` | `"zh-cn"` | 必须与文本语言一致，否则发音混乱 |
| `gpt_cond_latent` / `speaker_embedding` | 由参考音频提取 | 音色来源，见下方两步推理流程 |
| `temperature` | 0.65 | 生成随机性，过高吐字不稳、过低韵律呆板 |
| `top_p` / `top_k` | 0.5 / 50 | 与 temperature 配合控制采样分布 |
| `repetition_penalty` | 5.0 | 抑制重复字词卡壳（XTTS 官方推荐值） |
| `speed` | 1.0 | 语速倍率，0.8~1.3 之间自然 |
| `enable_text_splitting` | True | 长文本按句切分逐段生成，避免超长上下文崩坏 |

> 注意：新版 `Xtts.inference()` 不再直接接受 `speaker_wav` 参数，需先用
> `get_conditioning_latents()` 从参考音频提取「音色指纹」，再传入推理。

In [4]:
# Cell 4: 中文声音克隆推理（GPU 上约每 10 秒音频耗时 5~15 秒）
import time
import soundfile as sf

OUT_DIR = r"S:\PythonAILearningCourse\08-tts-voice-robot\02.coqui-tts"

# 第一步：从 18s 参考音频提取音色（GPT 条件 latent + 说话人向量），只需算一次
gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(audio_path=REF_WAV)
print("GPT 条件 latent:", tuple(gpt_cond_latent.shape), "| 说话人向量:", tuple(speaker_embedding.shape))

texts = [
    ("01_自我介绍.wav",
     "大家好，我是吴彦祖。欢迎来到人工智能语音课程，今天我用 XTTS 二代模型克隆自己的声音，为大家演示中文语音合成。"),
    ("02_长句测试.wav",
     "人工智能正在改变我们与声音打交道的方式。只需短短十几秒的参考音频，模型就能捕捉说话人的音色、韵律与情感，"
     "再输入任意一段中文文本，便能生成以假乱真的语音。"),
]

for fname, text in texts:
    t0 = time.time()
    out = model.inference(
        text=text,
        language="zh-cn",
        gpt_cond_latent=gpt_cond_latent,
        speaker_embedding=speaker_embedding,
        temperature=0.65,
        top_p=0.5,
        top_k=50,
        repetition_penalty=5.0,
        speed=1.0,
        enable_text_splitting=True,
    )
    wav = out["wav"]
    SR = config.model_args["output_sample_rate"]  # 模型真实输出采样率 24000
    sf.write(os.path.join(OUT_DIR, fname), wav, SR)
    print("%s | 文本 %d 字 | 音频 %.1f 秒 | 耗时 %.1f 秒" % (
        fname, len(text), len(wav) / SR, time.time() - t0))

GPT 条件 latent: (1, 32, 1024) | 说话人向量: (1, 512, 1)
01_自我介绍.wav | 文本 57 字 | 音频 11.3 秒 | 耗时 5.3 秒
02_长句测试.wav | 文本 76 字 | 音频 13.0 秒 | 耗时 4.0 秒


In [5]:
# Cell 5: 试听合成结果（Jupyter 内播放）
from IPython.display import Audio

Audio(os.path.join(OUT_DIR, "01_自我介绍.wav"))

## 4. 小结

- **XTTS-v2 = Speaker Encoder（提取音色）+ GPT（建模韵律）+ Vocoder（还原波形）**，零样本克隆只需一段短参考音频
- 本地加载三步：`XttsConfig.from_json` → `Xtts.init_from_config` → `model.load_checkpoint`，完全离线
- transformers 5.x 环境的兼容点：导入 TTS 前补上被移除的 `isin_mps_friendly`
- 参考音频越干净，克隆越像；中文文本务必传 `language="zh-cn"`

**可以继续尝试**：

1. 换参考音频（换成你自己的录音），听音色变化
2. 调整 `speed=0.9 / 1.15`，观察语速对自然度的影响
3. 把 `temperature` 拉到 1.0，体会吐字开始"飘"的边界
4. 结合上一章的 Gradio：上传参考音频 + 输入文本，做一个网页版声音克隆工具